<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/12A_GES_Aware_Genomic_RAG_Cell_7C5R_Structured_Output_API_Compatibility_Remediation_Authorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project folder is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact upstream identities, and remediation output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import copy
import csv
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd

NOTEBOOK_NAME = (
    '12A_GES_Aware_Genomic_RAG_Cell_7C5R_'
    'Structured_Output_API_Compatibility_Remediation_Authorization.ipynb'
)
CELL_ID = '7C5R'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_CELL_7C5_TERMINAL_DECISION = (
    'PASS_STAGE7C5_COMPLETE_CELL7C4_480_SCORE_BLIND_PROMPTS_AND_CELL7B4_FIXED_LLM_RUNTIME_'
    'CONFIG_REVERIFIED_1440_REQUEST_GENERATION_PLAN_FROZEN_CHECKSUM_PROTECTED_CELL7C6_'
    'EXACT_LLM_GENERATION_ONLY_AUTHORIZED_NO_SCORE_BEARING_AUDIT_CELL7A3_SCORES_'
    'ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

# Successful Cell 7C5 package.
CELL_7C5_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)
CELL_7C5_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)

CELL_7C5 = OrderedDict([
    ('authorization', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_stage7c_cell7c6_llm_generation_authorization_v1.json',
        'sha256': '8cb177929450933802e324ede4f00ffdb392c8d9f172b8692c28be28e3b81116',
    }),
    ('generation_plan', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_frozen_generation_request_plan_v1.parquet',
        'sha256': '6fe2a7c8dcd2601cfe10e269827a77f9cc78d0fd9f83dfad0644a0efd74b41cf',
    }),
    ('input_inventory', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_authorized_generation_input_inventory_v1.csv',
        'sha256': '8118dec995f09762c25f7ded6be9d77685a2b33d81fe05d7c2f6bd7e380f655d',
    }),
    ('qc', {
        'path': CELL_7C5_QC_DIR / 'cell_7c5_llm_generation_authorization_qc_v1.json',
        'sha256': '8f1eff5d5d6d39df5a32b709b235653a5a1dc26102850e875e673f72eef3900d',
    }),
    ('manifest', {
        'path': CELL_7C5_CONFIG_DIR / 'cell_7c5_llm_generation_authorization_manifest_v1.json',
        'sha256': '3bfb3a24fff0eb1ace3ff83d994c4f9ea4f0d0492ae014739378311fb48a51df',
    }),
])

# Exact Cell 7B4 config.
CELL_7B4_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'
CELL_7B4_LLM = {
    'path': CELL_7B4_DIR / 'cell_7b4_llm_prompt_response_configuration_v1.json',
    'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
}
CELL_7B4_RUNTIME = {
    'path': CELL_7B4_DIR / 'cell_7b4_runtime_and_determinism_configuration_v1.json',
    'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
}

# Failed Cell 7C6 checkpoint directory. The request-0 API rejection must have created no successful checkpoint.
CELL_7C6_CHECKPOINT_DIR = (
    ROOT / 'outputs' / 'execution_checkpoints' / 'stage7_rag'
    / 'cell_7c6_exact_llm_generation_v1'
)

AUTH_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c5r_structured_output_api_compatibility_remediation_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c5r_structured_output_api_compatibility_remediation_v1'
)

OUTPUTS = OrderedDict([
    ('api_compatible_schema',
     AUTH_DIR / 'cell_7c5r_api_compatible_response_schema_v1.json'),
    ('remediation_authorization',
     AUTH_DIR / 'cell_7c5r_stage7c_cell7c6_v2_generation_remediation_authorization_v1.json'),
    ('input_inventory',
     AUTH_DIR / 'cell_7c5r_remediation_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'cell_7c5r_structured_output_api_compatibility_qc_v1.json'),
    ('manifest',
     AUTH_DIR / 'cell_7c5r_structured_output_api_compatibility_manifest_v1.json'),
])

for directory in (AUTH_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(p) for p in OUTPUTS.values() if p.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C5R fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Remediation config directory: {AUTH_DIR}')
print(f'Remediation QC directory    : {QC_DIR}')

Remediation config directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c5r_structured_output_api_compatibility_remediation_v1
Remediation QC directory    : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c5r_structured_output_api_compatibility_remediation_v1


## 2. Strict checksum, sidecar, and serialization helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def canonical_json_text(payload: Any) -> str:
    return json.dumps(
        payload,
        sort_keys=True,
        separators=(',', ':'),
        ensure_ascii=False,
        allow_nan=False,
    )


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0]
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar format: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(
        payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + chr(10)
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(
        f'{digest}  {path.name}' + chr(10),
        encoding='utf-8',
    )


with tempfile.TemporaryDirectory(prefix='cell_7c5r_writer_test_') as tmp:
    test = Path(tmp) / 'x.json'
    stable_write_json(test, {'ok': True})
    write_sidecar(test)
    if json.loads(test.read_text(encoding='utf-8')) != {'ok': True}:
        raise AssertionError('JSON writer round-trip failed.')
    if not sidecar_is_valid(test):
        raise AssertionError('Sidecar writer/parser test failed.')

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify Cell 7C5 + Cell 7B4 and confirm request-0 failure produced no successful checkpoint

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C5.items():
    record = verify_exact_artifact(
        f'cell_7c5_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C5'
    verified_inputs.append(record)

for artifact_id, spec in [
    ('cell_7b4_llm_prompt_response', CELL_7B4_LLM),
    ('cell_7b4_runtime_determinism', CELL_7B4_RUNTIME),
]:
    record = verify_exact_artifact(artifact_id, spec['path'], spec['sha256'])
    record['source_cell'] = '7B4'
    verified_inputs.append(record)

manifest_7c5 = load_json(CELL_7C5['manifest']['path'])
qc_7c5 = load_json(CELL_7C5['qc']['path'])
llm_config = load_json(CELL_7B4_LLM['path'])

if manifest_7c5.get('terminal_decision') != EXPECTED_CELL_7C5_TERMINAL_DECISION:
    raise AssertionError('Cell 7C5 terminal decision mismatch.')
if manifest_7c5.get('next_authorized_cell') != '7C6':
    raise AssertionError('Cell 7C5 next authorized cell mismatch.')
if int(qc_7c5.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C5 QC does not report zero failures.')

checkpoint_jsons = (
    sorted(CELL_7C6_CHECKPOINT_DIR.glob('*.json'))
    if CELL_7C6_CHECKPOINT_DIR.exists()
    else []
)
checkpoint_sidecars = (
    sorted(CELL_7C6_CHECKPOINT_DIR.glob('*.json.sha256'))
    if CELL_7C6_CHECKPOINT_DIR.exists()
    else []
)

if checkpoint_jsons or checkpoint_sidecars:
    raise RuntimeError(
        'Cell 7C6 checkpoint directory is not empty. '
        'The reported failure was at 0/1440 before a successful response, so fail closed.\\n'
        f'Checkpoint JSON count: {len(checkpoint_jsons)}\\n'
        f'Checkpoint sidecar count: {len(checkpoint_sidecars)}'
    )

print('Cell 7C5 package                    : 5/5 VERIFIED')
print('Cell 7B4 LLM/runtime config         : VERIFIED')
print('Successful Cell 7C6 checkpoints    : 0')
print('Accepted LLM generation responses  : 0')
print('Answer keys loaded                  : NO')

Cell 7C5 package                    : 5/5 VERIFIED
Cell 7B4 LLM/runtime config         : VERIFIED
Successful Cell 7C6 checkpoints    : 0
Accepted LLM generation responses  : 0
Answer keys loaded                  : NO


## 4. Derive the one-key API-compatible schema projection and prove that nothing else changed

In [5]:
structured_cfg = llm_config['structured_output']
scientific_schema = copy.deepcopy(structured_cfg['schema'])

evidence_ids_schema = (
    scientific_schema
    .get('properties', {})
    .get('evidence_ids', {})
)

if evidence_ids_schema.get('type') != 'array':
    raise AssertionError('Frozen evidence_ids is no longer an array.')
if evidence_ids_schema.get('items') != {'type': 'string'}:
    raise AssertionError('Frozen evidence_ids item schema changed.')
if evidence_ids_schema.get('uniqueItems') is not True:
    raise AssertionError(
        'Expected exact frozen Cell 7B4 scientific schema to contain evidence_ids.uniqueItems=True.'
    )

scientific_schema_sha256 = sha256_text(canonical_json_text(scientific_schema))

# Narrow compatibility projection: remove exactly one unsupported JSON Schema keyword.
api_schema = copy.deepcopy(scientific_schema)
removed_value = api_schema['properties']['evidence_ids'].pop('uniqueItems')

if removed_value is not True:
    raise AssertionError('Unexpected removed uniqueItems value.')

api_schema_sha256 = sha256_text(canonical_json_text(api_schema))

# Prove only this exact path differs.
expected_reconstructed_scientific = copy.deepcopy(api_schema)
expected_reconstructed_scientific['properties']['evidence_ids']['uniqueItems'] = True

if expected_reconstructed_scientific != scientific_schema:
    raise AssertionError(
        'API-compatible schema differs from the frozen scientific schema by more than evidence_ids.uniqueItems.'
    )

# Preserve the scientific uniqueness contract post-response.
def scientific_evidence_ids_uniqueness_validator(payload: dict[str, Any]) -> bool:
    evidence_ids = payload.get('evidence_ids')
    return (
        isinstance(evidence_ids, list)
        and all(isinstance(value, str) for value in evidence_ids)
        and len(evidence_ids) == len(set(evidence_ids))
    )

remediation_checks = OrderedDict([
    ('scientific_schema_is_object', scientific_schema.get('type') == 'object'),
    ('scientific_uniqueItems_true',
     scientific_schema['properties']['evidence_ids'].get('uniqueItems') is True),
    ('api_uniqueItems_removed',
     'uniqueItems' not in api_schema['properties']['evidence_ids']),
    ('evidence_ids_type_preserved',
     api_schema['properties']['evidence_ids'].get('type') == 'array'),
    ('evidence_ids_items_preserved',
     api_schema['properties']['evidence_ids'].get('items') == {'type': 'string'}),
    ('root_required_preserved',
     api_schema.get('required') == scientific_schema.get('required')),
    ('root_additionalProperties_preserved',
     api_schema.get('additionalProperties') == scientific_schema.get('additionalProperties')),
    ('properties_keyset_preserved',
     set(api_schema.get('properties', {})) == set(scientific_schema.get('properties', {}))),
    ('only_one_keyword_removed',
     expected_reconstructed_scientific == scientific_schema),
    ('scientific_and_api_hashes_differ',
     scientific_schema_sha256 != api_schema_sha256),
    ('posthoc_uniqueness_validator_true_case',
     scientific_evidence_ids_uniqueness_validator({'evidence_ids': ['A', 'B']})),
    ('posthoc_uniqueness_validator_false_case',
     not scientific_evidence_ids_uniqueness_validator({'evidence_ids': ['A', 'A']})),
])

failed = [name for name, passed in remediation_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C5R compatibility derivation failed:\\n- '
        + '\\n- '.join(failed)
    )

print(f'Original scientific schema SHA-256 : {scientific_schema_sha256}')
print(f'API-compatible schema SHA-256      : {api_schema_sha256}')
print('Removed JSON Schema keyword        : properties.evidence_ids.uniqueItems')
print('Removed value                      : true')
print('Other schema changes               : NONE')
print('Post-response uniqueness validator : REQUIRED')

Original scientific schema SHA-256 : a32bbf83a5f954d64c60ee4d737299f86be80692aab773c35ebd6f681f3aa533
API-compatible schema SHA-256      : 527ab2c8b4e76974555b4c8cba99188e00b36a83334c4e190203f46b06238ef0
Removed JSON Schema keyword        : properties.evidence_ids.uniqueItems
Removed value                      : true
Other schema changes               : NONE
Post-response uniqueness validator : REQUIRED


## 5. Freeze remediation authorization for Cell 7C6 V2 only

In [6]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C6_V2_EXACT_1440_FROZEN_LLM_GENERATION_REQUESTS_'
    'WITH_API_COMPATIBLE_STRUCTURED_OUTPUT_SCHEMA_PROJECTION_REMOVING_ONLY_'
    'EVIDENCE_IDS_UNIQUEITEMS_AND_PRESERVING_POSTHOC_UNIQUENESS_VALIDATION_'
    'NO_PROMPT_MODEL_GENERATION_SETTING_SCORE_ANSWER_KEY_OR_EVALUATION_CHANGE'
)

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'remediation_type': 'api_schema_compatibility_only',
    'observed_failure': {
        'accepted_generation_requests_before_failure': 0,
        'error_class': 'BadRequestError',
        'http_status': 400,
        'api_message': (
            "Invalid schema for response_format 'ges_rag_genomic_evidence_answer': "
            "In context=('properties', 'evidence_ids'), 'uniqueItems' is not permitted."
        ),
        'scientific_response_observations_created': 0,
    },
    'original_scientific_schema': {
        'source_path': str(CELL_7B4_LLM['path']),
        'source_file_sha256': CELL_7B4_LLM['sha256'],
        'schema_sha256_canonical': scientific_schema_sha256,
        'evidence_ids_uniqueItems': True,
    },
    'api_compatibility_projection': {
        'api_schema_sha256_canonical': api_schema_sha256,
        'removed_path': 'properties.evidence_ids.uniqueItems',
        'removed_value': True,
        'number_of_schema_keyword_changes': 1,
        'field_set_changed': False,
        'required_fields_changed': False,
        'types_changed': False,
        'enums_changed': False,
        'numeric_bounds_changed': False,
        'descriptions_changed': False,
        'additionalProperties_changed': False,
    },
    'scientific_contract_preservation': {
        'evidence_ids_uniqueness_still_required': True,
        'enforcement_location': 'post-response validation before structured_valid=True',
        'validator_rule': 'len(evidence_ids) == len(set(evidence_ids))',
        'schema_invalid_or_duplicate_ids_are_preserved_as_observed_generation_outcomes': True,
        'preferred_answer_retry_prohibited': True,
    },
    'unchanged_frozen_design': {
        'prompts': 480,
        'runs_per_prompt': 3,
        'planned_requests': 1440,
        'model_snapshot': 'gpt-4.1-mini-2025-04-14',
        'temperature': 0.0,
        'top_p': 1.0,
        'max_output_tokens': 1200,
        'run_ids': [0, 1, 2],
        'prompt_text_changed': False,
        'context_membership_changed': False,
        'context_order_changed': False,
        'model_changed': False,
        'generation_settings_changed': False,
    },
    'authorization_decision': authorization_decision,
    'authorized_next_cell': '7C6_V2',
    'answer_key_access_authorized': False,
    'condition_unblinding_authorized': False,
    'adjudication_authorized': False,
    'evaluation_authorized': False,
    'llm_called_in_cell_7c5r': False,
}

stable_write_json(OUTPUTS['api_compatible_schema'], api_schema)
write_sidecar(OUTPUTS['api_compatible_schema'])

stable_write_json(OUTPUTS['remediation_authorization'], authorization_payload)
write_sidecar(OUTPUTS['remediation_authorization'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'remediation_checks': {
        name: bool(value) for name, value in remediation_checks.items()
    },
    'accepted_generation_requests_before_failure': 0,
    'successful_checkpoints_before_remediation': 0,
    'llm_called_in_remediation': False,
    'answer_key_accessed': False,
    'evaluation_performed': False,
    'failed_checks': 0,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

terminal_decision = (
    'PASS_STAGE7C5R_OPENAI_STRUCTURED_OUTPUT_SCHEMA_COMPATIBILITY_REMEDIATION_'
    'ORIGINAL_CELL7B4_SCHEMA_REVERIFIED_ONLY_EVIDENCE_IDS_UNIQUEITEMS_REMOVED_'
    'FOR_API_SERIALIZATION_POSTHOC_UNIQUENESS_VALIDATION_PRESERVED_CHECKSUM_PROTECTED_'
    'CELL7C6_V2_EXACT_1440_GENERATION_ONLY_AUTHORIZED_NO_OTHER_SCIENTIFIC_CHANGE'
)

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'upstream_lineage': {
        'cell_7c5_manifest_sha256': CELL_7C5['manifest']['sha256'],
        'cell_7c5_generation_plan_sha256': CELL_7C5['generation_plan']['sha256'],
        'cell_7b4_llm_config_sha256': CELL_7B4_LLM['sha256'],
        'cell_7b4_runtime_config_sha256': CELL_7B4_RUNTIME['sha256'],
    },
    'schema_lineage': {
        'original_scientific_schema_sha256_canonical': scientific_schema_sha256,
        'api_compatible_schema_sha256_canonical': api_schema_sha256,
        'only_removed_keyword': 'properties.evidence_ids.uniqueItems',
        'posthoc_uniqueness_validation_required': True,
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C6_V2',
    'answer_key_access_authorized': False,
    'evaluation_authorized': False,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Final readback.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C5R artifact readback failed: {path}')

auth_rb = load_json(OUTPUTS['remediation_authorization'])
schema_rb = load_json(OUTPUTS['api_compatible_schema'])
qc_rb = load_json(OUTPUTS['qc'])
manifest_rb = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('auth_decision_exact',
     auth_rb.get('authorization_decision') == authorization_decision),
    ('api_schema_uniqueItems_absent',
     'uniqueItems' not in schema_rb['properties']['evidence_ids']),
    ('api_schema_hash_exact',
     sha256_text(canonical_json_text(schema_rb)) == api_schema_sha256),
    ('qc_zero_failures',
     int(qc_rb.get('failed_checks', -1)) == 0),
    ('manifest_terminal_exact',
     manifest_rb.get('terminal_decision') == terminal_decision),
    ('manifest_next_7c6_v2',
     manifest_rb.get('next_authorized_cell') == '7C6_V2'),
    ('manifest_answer_keys_false',
     manifest_rb.get('answer_key_access_authorized') is False),
    ('manifest_evaluation_false',
     manifest_rb.get('evaluation_authorized') is False),
    ('all_sidecars_valid',
     all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C5R final readback failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(remediation_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C5R')
print('STRUCTURED-OUTPUT API COMPATIBILITY REMEDIATION AUTHORIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nFAILED CELL 7C6 REQUEST-0 BOUNDARY')
print('Accepted generation requests                  : 0 / 1,440')
print('Successful checkpoints                        : 0')
print('Model response generated                      : NO')
print('Failure class                                 : HTTP 400 invalid_json_schema')
print('Rejected keyword                              : evidence_ids.uniqueItems')

print('\\nNARROW COMPATIBILITY PROJECTION')
print(f'Original scientific schema SHA-256            : {scientific_schema_sha256}')
print(f'API-compatible schema SHA-256                 : {api_schema_sha256}')
print('Schema keyword removals                       : 1')
print('Removed                                      : properties.evidence_ids.uniqueItems = true')
print('Fields/types/enums/required/bounds changed    : NO')
print('Prompt/model/generation settings changed      : NO')
print('Evidence-ID uniqueness still required         : YES — post-response validator')

print('\\nCELL 7C6 V2 AUTHORIZATION')
print('Planned generation requests                   : 1,440')
print('API-compatible strict schema                  : AUTHORIZED')
print('Post-response evidence_ids uniqueness check   : MANDATORY')
print('Answer-key access                             : PROHIBITED')
print('Unblinding / adjudication / evaluation        : PROHIBITED')

print('\\nCELL 7C5R FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')
print('LLM called in Cell 7C5R                       : NO')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C5R
STRUCTURED-OUTPUT API COMPATIBILITY REMEDIATION AUTHORIZATION
Notebook                                      : 12A_GES_Aware_Genomic_RAG_Cell_7C5R_Structured_Output_API_Compatibility_Remediation_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nFAILED CELL 7C6 REQUEST-0 BOUNDARY
Accepted generation requests                  : 0 / 1,440
Successful checkpoints                        : 0
Model response generated                      : NO
Failure class                                 : HTTP 400 invalid_json_schema
Rejected keyword                              : evidence_ids.uniqueItems
\nNARROW COMPATIBILITY PROJECTION
Original scientific schema SHA-256            : a32bbf83a5f954d64c60ee4d737299f86be80692aab773c35ebd6f681f3aa533
AP